In [1]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import numpy as np

df = pd.read_csv('../backend/data/clean/dane_gotowe_ML.csv')

df.head()

,Cena,Przebieg,Pojemność skokowa,Moc,Wiek,Rodzaj paliwa_Benzyna+LPG,Rodzaj paliwa_Diesel,Rodzaj paliwa_Elektryczny,Rodzaj paliwa_Hybryda,Rodzaj paliwa_Hybryda Plug-in,...,Marka_Peugeot,Marka_Porsche,Marka_Renault,Marka_Seat,Marka_Skoda,Marka_Subaru,Marka_Suzuki,Marka_Toyota,Marka_Volkswagen,Marka_Volvo
0,327809,1,0,292,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,59000,118760,1969,254,6,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,80000,120000,1998,252,7,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,239000,22,0,1020,4,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,74900,5,1499,136,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [2]:
korelacje = df.corr()['Cena'].sort_values(ascending=False)
print("\n--- TOP CECHY WPŁYWAJĄCE NA CENĘ ---")
print(korelacje.head(10)) 
print("\n--- CECHY OBNIŻAJĄCE CENĘ ---")
print(korelacje.tail(5))


--- TOP CECHY WPŁYWAJĄCE NA CENĘ ---
Cena                             1.000000
Moc                              0.630075
Pojemność skokowa                0.305985
Marka_Mercedes-Benz              0.219666
Rodzaj paliwa_Hybryda Plug-in    0.204107
Marka_Porsche                    0.195889
Marka_BMW                        0.144447
Marka_Inne                       0.115013
Marka_Land                       0.114031
Rodzaj paliwa_Elektryczny        0.105431
Name: Cena, dtype: float64

--- CECHY OBNIŻAJĄCE CENĘ ---
Rodzaj paliwa_Benzyna+LPG   -0.110285
Marka_Opel                  -0.143083
Skrzynia biegów_Manualna    -0.508951
Przebieg                    -0.583136
Wiek                        -0.611414
Name: Cena, dtype: float64


In [3]:
X = df.drop(columns=['Cena'])
y = df['Cena']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

prognozy_cen = model.predict(X_test)

mae = mean_absolute_error(y_test, prognozy_cen)
rmse = np.sqrt(mean_squared_error(y_test, prognozy_cen)) 
r2 = r2_score(y_test, prognozy_cen)

print("WYNIKI MODELU")
print(f"Średni Błąd (MAE): {mae:,.0f} PLN (O tyle średnio model myli się na aucie)")
print(f"Błąd Ważony (RMSE): {rmse:,.0f} PLN (Kara za drastyczne pomyłki w drogich autach)")
print(f"Skuteczność (R²): {r2 * 100:.1f} % (Jaki % zmienności cen ogarnia model)")

WYNIKI MODELU
Średni Błąd (MAE): 30,888 PLN (O tyle średnio model myli się na aucie)
Błąd Ważony (RMSE): 50,005 PLN (Kara za drastyczne pomyłki w drogich autach)
Skuteczność (R²): 70.5 % (Jaki % zmienności cen ogarnia model)


In [4]:
print("Test na 5 pierwszych autach z zbioru testowego")
wyniki_test = pd.DataFrame({
    'Cena Prawdziwa (Otomoto)': y_test.values,
    'Wycena Modelu AI': prognozy_cen,
    'Pomyłka (Błąd)': np.abs(y_test.values - prognozy_cen)
})

pd.options.display.float_format = '{:,.0f}'.format
print(wyniki_test.head(3))

Test na 5 pierwszych autach z zbioru testowego
   Cena Prawdziwa (Otomoto)  Wycena Modelu AI  Pomyłka (Błąd)
0                     58000           112,237          54,237
1                      6500           -11,815          18,315
2                    116900           121,370           4,470


In [5]:
kolumny_numeryczne = ['Przebieg', 'Pojemność skokowa', 'Moc', 'Wiek']

num_transformer = Pipeline(steps=[
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()) 
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, kolumny_numeryczne)
    ],
    remainder='passthrough'
)

model_poly = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

print("Trenowanie modelu...")
model_poly.fit(X_train, y_train)

prognozy_poly = model_poly.predict(X_test)

mae_poly = mean_absolute_error(y_test, prognozy_poly)
rmse_poly = np.sqrt(mean_squared_error(y_test, prognozy_poly))
r2_poly = r2_score(y_test, prognozy_poly)

print(f"Średni Błąd (MAE): {mae_poly:,.0f} PLN")
print(f"Błąd Ważony (RMSE): {rmse_poly:,.0f} PLN")
print(f"Skuteczność (R²): {r2_poly * 100:.1f} %")

Trenowanie modelu...
Średni Błąd (MAE): 21,842 PLN
Błąd Ważony (RMSE): 38,138 PLN
Skuteczność (R²): 82.8 %


In [6]:
from sklearn.ensemble import GradientBoostingRegressor

gbt_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,        
    max_depth=4,              
    min_samples_leaf=2,
    random_state=42)

gbt_model.fit(X_train, y_train)

y_pred_gbt = gbt_model.predict(X_test)

mae_gbt = mean_absolute_error(y_test, y_pred_gbt)
rmse_gbt = np.sqrt(mean_squared_error(y_test, y_pred_gbt))
r2_gbt = r2_score(y_test, y_pred_gbt)

print("\n WYNIKI MODELU GRADIENT BOOSTING")
print(f"Skuteczność na teście (R²):   {r2_gbt * 100:.1f} %")
print(f"Średni Błąd (MAE):             {mae_gbt:,.0f} PLN")
print(f"Błąd Ważony (RMSE):            {rmse_gbt:,.0f} PLN")

importances_gbt = gbt_model.feature_importances_
feature_names = X_train.columns

df_importances_gbt = pd.DataFrame({'Cecha': feature_names, 'Ważność': importances_gbt})
df_importances_gbt = df_importances_gbt.sort_values(by='Ważność', ascending=False)

print("\n TOP 5 NAJWAŻNIEJSZYCH CECH (GBT)")
print(df_importances_gbt.head(5).to_string(index=False))


 WYNIKI MODELU GRADIENT BOOSTING
Skuteczność na teście (R²):   89.5 %
Średni Błąd (MAE):             15,425 PLN
Błąd Ważony (RMSE):            29,889 PLN

 TOP 5 NAJWAŻNIEJSZYCH CECH (GBT)
               Cecha  Ważność
                Wiek        0
                 Moc        0
   Pojemność skokowa        0
            Przebieg        0
Rodzaj paliwa_Diesel        0


Poprzedni model StackingRegressor został porzucony ze względu na bardzo duży rozmiar, co przy ewentualnej chęci wrzucenia na produkcję na darmowy hosting, nie jest wskazane.
Zatem StackingRegressor dalej pozostanie użyty, ale bez RandomForest, który powodował drastyczny wzrost wagi modelu.

In [7]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

estimators = [
    ('poly', model_poly),  
    ('gbt', gbt_model)    
]


stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression(),
    n_jobs=-1  
)

print("Trenowanie stacku...")
stacking_model.fit(X_train, y_train)

y_pred_stack = stacking_model.predict(X_test)

mae_stack = mean_absolute_error(y_test, y_pred_stack)
rmse_stack = np.sqrt(mean_squared_error(y_test, y_pred_stack))
r2_stack = r2_score(y_test, y_pred_stack)

print("\nWYNIKI MODELU STACKING ")
print(f"Skuteczność na teście (R²):   {r2_stack * 100:.1f} %")
print(f"Średni Błąd (MAE):             {mae_stack:,.0f} PLN")
print(f"Błąd Ważony (RMSE):            {rmse_stack:,.0f} PLN")

Trenowanie stacku...

WYNIKI MODELU STACKING 
Skuteczność na teście (R²):   89.5 %
Średni Błąd (MAE):             15,405 PLN
Błąd Ważony (RMSE):            29,840 PLN


In [8]:
print("Test na 5 pierwszych autach z zbioru testowego")
wyniki_test = pd.DataFrame({
    'Cena Prawdziwa (Otomoto)': y_test.values,
    'Wycena Modelu AI': y_pred_stack,
    'Pomyłka (Błąd)': np.abs(y_test.values - y_pred_stack)
})

pd.options.display.float_format = '{:,.0f}'.format
print(wyniki_test.head(5))

Test na 5 pierwszych autach z zbioru testowego
   Cena Prawdziwa (Otomoto)  Wycena Modelu AI  Pomyłka (Błąd)
0                     58000            79,075          21,075
1                      6500            15,205           8,705
2                    116900           148,120          31,220
3                     22900            20,626           2,274
4                    219900           247,616          27,716


In [ ]:
import pandas as pd

auto_parametry = {
    'Przebieg': 160000,
    'Pojemność skokowa': 1600,
    'Moc': 116,
    'Wiek': 16, 
    'Rodzaj paliwa_Benzyna': 1,
    'Skrzynia biegów_Manualna': 1,
    'Marka_Mitsubishi': 1 
}

nowe_auto_df = pd.DataFrame(0, index=[0], columns=X_train.columns)

for cecha, wartosc in auto_parametry.items():
    if cecha in nowe_auto_df.columns:
        nowe_auto_df.at[0, cecha] = wartosc
    elif cecha.startswith('Marka_'):
        if 'Marka_Inne' in nowe_auto_df.columns:
            nowe_auto_df.at[0, 'Marka_Inne'] = 1

wycena = stacking_model.predict(nowe_auto_df)[0]

print(f"Model auta: Mitsubishi ASX 1.6 Benzyna, Manual, 2010 (160 tys. km)")
print(f"Szacowana wartość: {wycena:,.0f} PLN")

Model auta: Mitsubishi ASX 1.6 Benzyna, Manual, 2010 (160 tys. km)
Szacowana wartość: 20,699 PLN


In [11]:
import joblib

joblib.dump(stacking_model, '../backend/data/light_car_price_model_500_pages.pkl')

kolumny_treningowe = list(X_train.columns)
joblib.dump(kolumny_treningowe, '../backend/data/light_model_columns_500_pages.pkl')

['../backend/data/light_model_columns_500_pages.pkl']